# Upsert

In [0]:
import pyspark.sql.functions as F
import pyspark.sql.types as T
from delta.tables import DeltaTable

In [0]:
schema = T.StructType([
    T.StructField("Name", T.StringType(), True),
    T.StructField("Last_Name", T.StringType(), True),
    T.StructField("Age", T.IntegerType(), True)
])

data = [("Alicia", "Rojas", 25), ("Pedro", "Vargas", 30), ("Rafael", "Quintero", 30)]
df = spark.createDataFrame(data, schema=schema)

In [0]:
df.display()

In [0]:
uc_table = "momentum.bronze.persons"
df.write.mode("overwrite").saveAsTable(uc_table)

In [0]:
data = [("Alicia", "Rojas", 28), ("Pedro", "Vargas", 30), ("Rafael", "Quintero", 30), ("María", "Fonseca", 23)]
df_uptated = spark.createDataFrame(data, schema=schema)

In [0]:
df_updated.write.mode("overwrite").saveAsTable(uc_table)

In [0]:
data = [("Alicia", "Rojas", 28), ("Pedro", "Vargas", 30), ("María", "Fonseca", 23), ("Enrique", "Morales", 32)]
df_uptated = spark.createDataFrame(data, schema=schema)

In [0]:
# ¿Qué pasa si no quieres perder los registros que no se encuentran en la últimaa versión?

In [0]:
# deltaTable = DeltaTable.forName(spark, uc_table)

# (deltaTable.alias("target")
#  .merge(df.alias("source"), "target.id = source.id")
#  .whenMatchedUpdate(set={"name": "source.name", "value": "source.value"})
#  .whenNotMatchedInsert(values={"id": "source.id", "name": "source.name", "value": "source.value"})
#  .execute())

In [0]:
%sql
MERGE INTO momentum.bronze.target_table AS target
USING df_access AS source
ON target.id = source.id
WHEN MATCHED THEN
  UPDATE SET target.name = source.name, target.value = source.value
WHEN NOT MATCHED THEN
  INSERT (id, name, value) VALUES (source.id, source.name, source.value);